In [1]:
import pandas as pd
import glob
import os
import re

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [2]:
#gather all absolute file paths in raw data folder and place them in a list
def get_files():
    raw_data_path = os.path.abspath(os.path.join(os.getcwd(),".."))
    pattern = os.path.join(raw_data_path,"data","raw","*.xlsx")
    file_lst = glob.glob(pattern)
    return file_lst

In [3]:
#clean and filter data to acceptable parameters
def clean_filter(quarter_df_lst):
    #strip and force lowercase on all, force single "_" between words 
    for q in quarter_df_lst:
        q.columns = q.columns.str.lower()
        q.columns = q.columns.str.strip()
        q.columns = q.columns.str.replace('\\s+', '_', regex=True)
    #filter out any sheets with less than 5 rows
    length_check = [v for v in quarter_df_lst if len(v) > 5]
    #filter out sheets without expected cols
    required_cols = {"start","details","status_text","booking_source","player_count","tee_sheet","date_cancelled"}
    col_check = [v for v in length_check if required_cols.issubset(set(v.columns))]
    #filter out sheets with more than 10% NA values on the start time
    valid_start_time = [v for v in col_check if v["start"].isna().mean() < 0.10]
    final = valid_start_time
    return final
    
#for each file, collect each tab into a dict w/ tab name as key, df of data as value. concat all tabs into a full year df
year_dfs = []
file_lst = get_files()
for file in file_lst:
    quarter_dict = pd.read_excel(file,sheet_name=None)
    quarters = list(quarter_dict.values())
    quarters = clean_filter(quarters)
    if quarters:
        year = pd.concat(quarters)
        year_dfs.append(year)
    else: 
        print(f"'{os.path.basename(file)}': no valid data")

#concat all year dfs into one master df
master_raw = pd.concat(year_dfs,ignore_index=True)


In [4]:
#print(master_raw.dtypes)

In [5]:
#basic cleaning, convert start to datetime, drop booking_source (provides no meaningful input here), rename start to tee_time for clarity
#convert cancellation time to datetime
master_df = master_raw.copy()
master_df["start"] = pd.to_datetime(master_df["start"])
master_df.drop(columns=["booking_source"],inplace=True)
master_df.rename(columns={"start":"tee_time"},inplace=True)
master_df["date_cancelled"] = pd.to_datetime(master_df["date_cancelled"],errors="coerce")
display(master_df)

,tee_time,details,status_text,player_count,tee_sheet,date_cancelled
0,2021-03-14 07:51:00,Reserved using online booking @ 7:00pm 3/7 EST Amount due at course: $124.00,checked in,2,Bethpage Early AM 9 Holes Blue,NaT
1,2021-03-14 07:51:00,Reserved using online booking @ 9:35pm 3/13 EST Original amount due at course: 1 Players for $31.00,checked in,1,Bethpage Early AM 9 Holes Blue,NaT
2,2021-03-14 08:00:00,Reserved using online booking @ 3:32pm 3/13 EST Original amount due at course: 2 Players for $62.00,checked in,2,Bethpage Early AM 9 Holes Blue,NaT
3,2021-03-14 08:27:00,Reserved using online booking @ 11:19am 3/13 EST Original amount due at course: 4 Players for $124.00,checked in,4,Bethpage Early AM 9 Holes Blue,NaT
4,2021-03-14 08:36:00,Reserved using online booking @ 5:42pm 3/13 EST Original amount due at course: 3 Players for $93.00,checked in,1,Bethpage Early AM 9 Holes Blue,NaT
...,...,...,...,...,...,...
616027,2025-10-21 13:42:00,Reserved using online booking @ 8:00pm 10/14 EDT Original amount due at course: 1 Players for $15.00 - PAID BOOKING FEE OF $5.00,NaN,1,Bethpage Blue Course,NaT
616028,2025-10-21 13:42:00,Reserved using online booking @ 12:32pm 10/15 EDT Original amount due at course: 1 Players for $23.00 - PAID BOOKING FEE OF $5.00,NaN,1,Bethpage Blue Course,NaT
616029,2025-10-21 13:51:00,Reserved using online booking @ 7:02pm 10/14 EDT Original amount due at course: 4 Players for $92.00 - PAID BOOKING FEE OF $20.00,NaN,4,Bethpage Blue Course,NaT
616030,2025-10-21 14:00:00,Reserved using online booking @ 8:55pm 10/14 EDT,deleted,4,Bethpage Blue Course,2025-10-14 21:21:01


In [6]:
# #evaluating unique values and counts to see if it makes sense apply some further cleaning to allow more analysis
# for col in master_df.columns:
#     display(master_df[col].value_counts(dropna=False))
#     display(master_df.loc[master_df[col].isna()])

In [7]:
#clean time, date, and cost out of details using regex,drop any rows that extract time data to nan
master_df[["book_hr","book_min","am_pm","book_month","book_day","cost_per_group"]] = master_df["details"].str.extract(r"@\s(\d{1,2}):(\d{2})(am|pm|AM|PM|Am|Pm)\s(\d{1,2})\/(\d{1,2})(?:.*?\$(\d+\.\d{2}))?")
failed_extract_count = master_df[["book_hr","book_min","am_pm","book_month","book_day"]].isna().any(axis=1).sum()
master_df.dropna(subset=["book_hr","book_min","am_pm","book_month","book_day"],inplace=True)
master_df[["book_hr","book_min","book_month","book_day"]] = master_df[["book_hr","book_min","book_month","book_day"]].astype(int)

#reformat book time to 24hr to enable cleaning
master_df.loc[(master_df["am_pm"] == "pm")&(master_df["book_hr"] != 12), "book_hr"] = master_df.loc[(master_df["am_pm"] == "pm")&(master_df["book_hr"] != 12), "book_hr"]+12
master_df.loc[(master_df["book_hr"] == 12) & (master_df["am_pm"] == "am"), "book_hr"] = 0

# pull year from start, adjust booking time to previous year if booking in dec and tee time in jan
master_df["book_yr"] = master_df["tee_time"].dt.year
master_df.loc[(master_df["tee_time"].dt.month == 1)&(master_df["book_month"] == 12),"book_yr"] = master_df.loc[(master_df["tee_time"].dt.month == 1)&(master_df["book_month"] == 12),"book_yr"]-1

#check that date components are within valid ranges
date_check = {
    "months": (master_df["book_month"].between(1, 12)).all(),
    "days": (master_df["book_day"].between(1, 31)).all(),
    "hrs": (master_df["book_hr"].between(0, 23)).all(),
    "mins": (master_df["book_min"].between(0, 59)).all(),
}

#convert cleaned data to datetime
master_df["booking_time"] = pd.to_datetime(
    {
        "year": master_df["book_yr"],
        "month": master_df["book_month"],
        "day": master_df["book_day"],
        "hour": master_df["book_hr"],
        "minute": master_df["book_min"],
    },
    errors="coerce",
)
book_time_failures_count = master_df["booking_time"].isna().sum()
bad_book_time = master_df.index[master_df["booking_time"].isna()]
master_df.drop(bad_book_time,inplace=True)

# #drop helper date columns to clean up df, reorganize cols
master_df.drop(columns=["book_hr","book_min","am_pm","book_month","book_day", "book_yr"],inplace=True)
master_df = master_df[['tee_time', 'details', 'booking_time', 'status_text', 'player_count', 'tee_sheet',
       'date_cancelled', 'cost_per_group']]

#drop any rows where booking time occurs after tee time. No clear explanation for this upon analysis so it will be treated as noise (~1 row impacted)
bad_time_order = master_df.index[master_df["booking_time"] > master_df["tee_time"]]
master_df.drop(bad_time_order,inplace=True)


In [8]:
#details col extraction QA checks:
print("Details column data extraction QA checks:")
print(f"\t{failed_extract_count} rows dropped due to invalid time data regex results")
for k,v in date_check.items():
    if v:
        print(f"\tall {k} in valid ranges")
    else:
        print(f"\t{k} has invalid values")
print(f"\t{book_time_failures_count} rows dropped due to failed booking time datetime conversion")

Details column data extraction QA checks:
	4 rows dropped due to invalid time data regex results
	all months in valid ranges
	all days in valid ranges
	all hrs in valid ranges
	all mins in valid ranges
	0 rows dropped due to failed booking time datetime conversion


In [9]:
# #status_text investigation and cleaning
# for val in master_df["status_text"].unique():
#     display(master_df.loc[master_df["status_text"] == val])
# display(master_df.loc[master_df["status_text"].isna()])

#Adjust and collapse status_text values based on invesigation
#collapse "checked in" and "teed off" to "played"
df_size_start = len(master_df)
master_df.loc[(master_df["status_text"] == "checked in")|(master_df["status_text"] == "teed off"),"status_text"] = "played"

#reword deleted to cancelled for clarity 
master_df.loc[master_df["status_text"] == "deleted","status_text"] = "cancelled"

#coerce NAN to unknown
master_df.loc[master_df["status_text"].isna(),"status_text"] = "unknown"

#QA helper values
df_size_end = len(master_df)
played_cancel_time_count = ((master_df["status_text"]=="played")&(master_df["date_cancelled"].notna())).sum()
cancel_no_time_count = ((master_df["status_text"]=="cancelled")&(master_df["date_cancelled"].isna())).sum()
#display(master_df.loc[(master_df["status_text"]=="cancelled")&(master_df["date_cancelled"].isna())])

In [10]:
#status_text col extraction QA checks:
print("status_text column data extraction QA checks:")
if df_size_start == df_size_end:
    print(f"\tdataframe size remained constant")
else:
    print(f"\tdataframe size did not remain constant")
print(f"\t{played_cancel_time_count} rows marked as played but have a cancellation time listed")
print(f"\t{cancel_no_time_count} rows marked as cancelled but have no cancellation time listed")

status_text column data extraction QA checks:
	dataframe size remained constant
	0 rows marked as played but have a cancellation time listed
	800 rows marked as cancelled but have no cancellation time listed


In [11]:
#player_count cleaning
# invalid_player_counts = master_df.loc[(master_df["player_count"]<1)|(master_df["player_count"]>4)].sort_values(by="player_count")
# display(invalid_player_counts)
#extract player counts from details column and count mismatches between native player_count data
master_df["player_count_regex"] = master_df["details"].str.extract(r"([1-4])\s+Players?").astype(float)
discrepant_row_init_pct = (master_df["player_count"] != master_df["player_count_regex"]).mean()*100

#create new col to capture reconsiled value and correct bad values in player_count with regex results
master_df["player_count_final"] = master_df["player_count"]
master_df.loc[(master_df["player_count_final"]!=master_df["player_count_regex"])&(master_df["player_count_regex"].notna()),"player_count_final"] = master_df["player_count_regex"]

#recount rows with discrepancies after cleaning and drop them since no valid data can be extracted to correct them
discrepant_row_fin_pct = ((master_df["player_count_final"]<1)|(master_df["player_count_final"]>4)|(master_df["player_count_final"].isna())).mean()*100
bad_row_index = master_df.loc[(master_df["player_count_final"]<1)|(master_df["player_count_final"]>4)|(master_df["player_count_final"].isna())].index
master_df.drop(bad_row_index,inplace=True)

#drop player_count (exists in raw df still) and regex helper column to clean up df
master_df.drop(columns=["player_count","player_count_regex"],inplace=True)


In [12]:
#player_count QA
print("player_count column data extraction QA checks:")
print(f"\t{discrepant_row_init_pct:.2f}% of rows were corrected after finding incorrect player counts in player_count")
print(f"\t{discrepant_row_fin_pct:.4f}% of rows were dropped after finding incorrect player counts after cleaning")

player_count column data extraction QA checks:
	12.48% of rows were corrected after finding incorrect player counts in player_count
	0.0003% of rows were dropped after finding incorrect player counts after cleaning


In [13]:
#clean round length and course out of tee_sheet
master_df[["course","round_length"]] = master_df["tee_sheet"].str.extract(r"\b(black|red|blue|green|yellow)\b\s(\d?)")

ValueError: Cannot set a DataFrame with multiple columns to the single column round_length

In [14]:
#evaluating unique values and counts to see if it makes sense apply some further cleaning to allow more analysis
for col in master_df.columns:
    display(master_df[col].value_counts(dropna=False))
    display(master_df.loc[master_df[col].isna()])

tee_time
2024-06-03 06:39:00    47
2024-06-15 17:00:00    45
2024-06-10 06:03:00    40
2024-05-23 07:06:00    39
2024-04-13 08:27:00    39
                       ..
2025-10-21 12:30:00     1
2025-10-21 12:48:00     1
2025-10-21 12:57:00     1
2025-10-21 13:15:00     1
2025-10-20 08:36:00     1
Name: count, Length: 117813, dtype: int64

,tee_time,details,booking_time,status_text,tee_sheet,date_cancelled,cost_per_group,player_count_final


details
Reserved using online booking @ 7:00pm 6/27 EDT Original amount due at course: 4 Players for $192.00                                 266
Reserved using online booking @ 7:00pm 5/30 EDT Original amount due at course: 4 Players for $172.00                                 238
Reserved using online booking @ 7:00pm 6/21 EDT Original amount due at course: 4 Players for $172.00                                 235
Reserved using online booking @ 7:00pm 6/20 EDT Original amount due at course: 4 Players for $172.00                                 228
Reserved using online booking @ 7:00pm 6/12 EDT Original amount due at course: 4 Players for $192.00                                 226
                                                                                                                                    ... 
Reserved using online booking @ 7:41pm 10/14 EDT Original amount due at course: 2 Players for $86.00 - PAID BOOKING FEE OF $10.00      1
Reserved using online booking @ 8

,tee_time,details,booking_time,status_text,tee_sheet,date_cancelled,cost_per_group,player_count_final


booking_time
2022-05-21 19:00:00    233
2022-05-22 19:00:00    218
2022-07-10 19:00:00    216
2022-06-26 19:00:00    214
2023-06-09 19:00:00    213
                      ... 
2021-03-13 22:55:00      1
2021-03-13 23:18:00      1
2021-03-13 20:26:00      1
2021-03-13 20:37:00      1
2021-03-13 18:42:00      1
Name: count, Length: 374335, dtype: int64

...

,tee_time,details,booking_time,status_text,tee_sheet,date_cancelled,cost_per_group,player_count_final
0,2021-03-14 07:51:00,Reserved using online booking @ 7:00pm 3/7 EST Amount due at course: $124.00,2021-03-07 19:00:00,played,Bethpage Early AM 9 Holes Blue,NaT,124.00,2
1,2021-03-14 07:51:00,Reserved using online booking @ 9:35pm 3/13 EST Original amount due at course: 1 Players for $31.00,2021-03-13 21:35:00,played,Bethpage Early AM 9 Holes Blue,NaT,31.00,1
2,2021-03-14 08:00:00,Reserved using online booking @ 3:32pm 3/13 EST Original amount due at course: 2 Players for $62.00,2021-03-13 15:32:00,played,Bethpage Early AM 9 Holes Blue,NaT,62.00,2
3,2021-03-14 08:27:00,Reserved using online booking @ 11:19am 3/13 EST Original amount due at course: 4 Players for $124.00,2021-03-13 11:19:00,played,Bethpage Early AM 9 Holes Blue,NaT,124.00,4
4,2021-03-14 08:36:00,Reserved using online booking @ 5:42pm 3/13 EST Original amount due at course: 3 Players for $93.00,2021-03-13 17:42:00,played,Bethpage Early AM 9 Holes Blue,NaT,93.00,3
...,...,...,...,...,...,...,...,...
616026,2025-10-21 13:33:00,Reserved using online booking @ 7:41pm 10/14 EDT Original amount due at course: 2 Players for $86.00 - PAID BOOKING FEE OF $10.00,2025-10-14 19:41:00,unknown,Bethpage Red Course,NaT,86.00,2
616027,2025-10-21 13:42:00,Reserved using online booking @ 8:00pm 10/14 EDT Original amount due at course: 1 Players for $15.00 - PAID BOOKING FEE OF $5.00,2025-10-14 20:00:00,unknown,Bethpage Blue Course,NaT,15.00,1
616028,2025-10-21 13:42:00,Reserved using online booking @ 12:32pm 10/15 EDT Original amount due at course: 1 Players for $23.00 - PAID BOOKING FEE OF $5.00,2025-10-15 12:32:00,unknown,Bethpage Blue Course,NaT,23.00,1
616029,2025-10-21 13:51:00,Reserved using online booking @ 7:02pm 10/14 EDT Original amount due at course: 4 Players for $92.00 - PAID BOOKING FEE OF $20.00,2025-10-14 19:02:00,unknown,Bethpage Blue Course,NaT,92.00,4


cost_per_group
192.00    69824
172.00    67826
43.00     41383
48.00     41335
86.00     31698
          ...  
171.00        1
404.00        1
202.00        1
57.00         1
126.00        1
Name: count, Length: 138, dtype: int64

,tee_time,details,booking_time,status_text,tee_sheet,date_cancelled,cost_per_group,player_count_final
763,2021-03-23 07:42:00,Reserved using online booking @ 9:31am 3/20 EDT\nCustomer Message: Booked Using the IVR System,2021-03-20 09:31:00,played,Bethpage Blue Course,NaT,NaN,1
1270,2021-03-25 17:36:00,Reserved using online booking @ 2:57pm 3/23 EDT\nCustomer Message: Booked Using the IVR System,2021-03-23 14:57:00,cancelled,Bethpage Green Course,2021-03-23 13:07:46,NaN,1
3408,2021-04-06 15:39:00,Reserved using online booking @ 8:17am 4/3 EDT\nCustomer Message: Booked Using the IVR System,2021-04-03 08:17:00,cancelled,Bethpage Yellow 9 Hole Course,2021-04-03 07:01:59,NaN,4
3991,2021-04-08 17:09:00,Reserved using online booking @ 8:59am 4/7 EDT\nCustomer Message: Booked Using the IVR System,2021-04-07 08:59:00,played,Bethpage Yellow 9 Hole Course,NaT,NaN,2
5056,2021-04-14 09:03:00,Reserved using online booking @ 6:15am 4/13 EDT\nCustomer Message: Booked Using the IVR System,2021-04-13 06:15:00,played,Bethpage Green Course,NaT,NaN,1
...,...,...,...,...,...,...,...,...
615355,2025-10-13 09:03:00,Reserved using online booking @ 7:10pm 10/6 EDT,2025-10-06 19:10:00,cancelled,Bethpage Red Course,2025-10-10 17:24:01,NaN,4
615632,2025-10-16 09:03:00,Reserved using online booking @ 2:44pm 10/15 EDT - PAID ONLINE - SELF-CHECKIN,2025-10-15 14:44:00,played,Bethpage Early AM 9 Holes Blue,NaT,NaN,1
615653,2025-10-16 11:45:00,Reserved using online booking @ 7:28pm 10/9 EDT,2025-10-09 19:28:00,unknown,Bethpage Red Course,NaT,NaN,2
615966,2025-10-20 13:06:00,Reserved using online booking @ 10:31pm 10/13 EDT,2025-10-13 22:31:00,cancelled,Bethpage Blue Course,2025-10-13 22:56:01,NaN,2


player_count_final
4    262007
1    175931
2    143712
3     34363
Name: count, dtype: int64

,tee_time,details,booking_time,status_text,tee_sheet,date_cancelled,cost_per_group,player_count_final
